# ASAP-AES set 1: fine-tuned DistilBERT (pre-declared)

Runs `backend/scripts/asap_finetune.py` on a Colab T4. The design was pre-declared in `asap_results.md` (commit 494bd77) and is unchanged.

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells in order. **Step 7 scores the locked test split: run it once.** The script refuses a second run into the same output folder.

In [ ]:
# 1. Confirm the GPU
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Point at the uploaded folder and check it is all there
ROOT = "/content/drive/MyDrive/asap_finetune"
!ls "{ROOT}/backend/scripts" "{ROOT}/backend/app/grading" "{ROOT}/data" "{ROOT}/data/embedding_cache"

In [ ]:
# 4. Install dependencies (torch is Colab's own CUDA build; this normally installs nothing)
!pip install -q -r "{ROOT}/backend/requirements-colab.txt"

In [ ]:
# 5. Pre-flight only: loads the data, checks the locked split, and checks that the two
#    reference systems reproduce the published benchmark. No training. Expect: PRE-FLIGHT PASSED
!python "{ROOT}/backend/scripts/asap_finetune.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output" --dry-run

In [ ]:
# 6. Smoke test: every code path on 32 train essays, one epoch. A dev slice stands in
#    for test, so the locked test split is never read. Its numbers mean nothing.
!python "{ROOT}/backend/scripts/asap_finetune.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output_smoke" --smoke

In [ ]:
# 7. THE RUN -- run once. Trains the full grid, chooses on dev, scores test once.
!python "{ROOT}/backend/scripts/asap_finetune.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output"

In [ ]:
# 8. Confirm the outputs landed on Drive, then flush Drive so nothing is lost
!ls -la "{ROOT}/output" "{ROOT}/output/best_model"
drive.flush_and_unmount()
print("Done. Download output/asap_finetune.json and output/finetune_log.txt.")